In [10]:
import sqlite3

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import pearsonr

DB_PATH = "../data/pac5.db"
conn = sqlite3.connect(DB_PATH)

query = """
SELECT
    pm7_hl_gap,
    xtb_hl_gap,
    camb3lyp_hl_gap
FROM
    pac5
WHERE
    camb3lyp_hl_gap IS NOT NULL
    AND pm7_hl_gap IS NOT NULL
    AND xtb_hl_gap IS NOT NULL
    AND is_open_shell = 0
"""

df = pd.read_sql_query(query, conn)
conn.close()


correlation_pm7, _ = pearsonr(df["pm7_hl_gap"], df["camb3lyp_hl_gap"])
correlation_xtb, _ = pearsonr(df["xtb_hl_gap"], df["camb3lyp_hl_gap"])

print(f"Pearson Correlation (PM7 vs CAM-B3LYP): {correlation_pm7:.4f}")
print(f"Pearson Correlation (GFN2-xTB vs CAM-B3LYP): {correlation_xtb:.4f}")

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.1)

min_val = 0
max_val = np.ceil(max(df[["pm7_hl_gap", "xtb_hl_gap", "camb3lyp_hl_gap"]].max()))

fig.add_trace(
    go.Scattergl(
        x=df["camb3lyp_hl_gap"],
        y=df["pm7_hl_gap"],
        mode="markers",
        name="PM7",
        marker=dict(opacity=0.7),
        hovertemplate="CAM-B3LYP: %{x:.3f} eV<br>PM7: %{y:.3f} eV<extra></extra>",
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scattergl(
        x=df["camb3lyp_hl_gap"],
        y=df["xtb_hl_gap"],
        mode="markers",
        name="GFN2-xTB",
        marker=dict(opacity=0.7),
        hovertemplate="CAM-B3LYP: %{x:.3f} eV<br>GFN2-xTB: %{y:.3f} eV<extra></extra>",
    ),
    row=1,
    col=2,
)

for col in [1, 2]:
    fig.add_trace(
        go.Scatter(
            x=[min_val, max_val],
            y=[min_val, max_val],
            mode="lines",
            line=dict(dash="dash", color="black", width=2),
            name="Parity Line",
            showlegend=False,
        ),
        row=1,
        col=col,
    )

subplot_info = [
    (1, "a", correlation_pm7),
    (2, "b", correlation_xtb),
]

for col, letter, corr in subplot_info:
    fig.add_annotation(
        text=f"<b>{letter}</b>",
        xref="x domain",
        yref="y domain",
        x=0.02,
        y=0.98,
        xanchor="left",
        yanchor="top",
        showarrow=False,
        font=dict(size=28, color="black"),
        row=1,
        col=col,
    )

    fig.add_annotation(
        text=f" r = {corr:.3f} ",
        xref="x domain",
        yref="y domain",
        x=0.14,
        y=0.97,
        xanchor="left",
        yanchor="top",
        showarrow=False,
        bgcolor="rgba(255, 255, 255, 0.8)",
        bordercolor="black",
        font=dict(size=24, color="black"),
        row=1,
        col=col,
    )

# Square panels via constrain, scaleanchor and scaleratio
fig.update_xaxes(
    title_text="<b>CAM-B3LYP HL-Gap (eV)</b>",
    row=1,
    col=1,
    range=[min_val, max_val],
    linewidth=1,
    linecolor="black",
    mirror=True,
    ticks="inside",
    showline=True,
    constrain="domain",
)
fig.update_yaxes(
    title_text="<b>PM7 HL-Gap (eV)</b>",
    row=1,
    col=1,
    range=[min_val, max_val],
    linewidth=1,
    linecolor="black",
    mirror=True,
    ticks="inside",
    showline=True,
    scaleanchor="x",
    scaleratio=1,
    constrain="domain",
)
fig.update_xaxes(
    title_text="<b>CAM-B3LYP HL-Gap (eV)</b>",
    row=1,
    col=2,
    range=[min_val, max_val],
    linewidth=1,
    linecolor="black",
    mirror=True,
    ticks="inside",
    showline=True,
    constrain="domain",
)
fig.update_yaxes(
    title_text="<b>GFN2-xTB HL-Gap (eV)</b>",
    row=1,
    col=2,
    range=[min_val, max_val],
    linewidth=1,
    linecolor="black",
    mirror=True,
    ticks="inside",
    showline=True,
    scaleanchor="x2",
    scaleratio=1,
    constrain="domain",
)
fig.update_layout(
    template="plotly",
    font=dict(family="Helvetica", size=22),
    width=1200,
    height=600,
    showlegend=False,
)
fig.write_image("../figures/si_figure_2.png", scale=2)
fig.show()

Pearson Correlation (PM7 vs CAM-B3LYP): 0.9214
Pearson Correlation (GFN2-xTB vs CAM-B3LYP): 0.8517


In [7]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_true = df["camb3lyp_hl_gap"]
y_pm7 = df["pm7_hl_gap"]
y_xtb = df["xtb_hl_gap"]

corr_pm7, _ = pearsonr(y_pm7, y_true)
r2_pm7 = r2_score(y_true, y_pm7)
mae_pm7 = mean_absolute_error(y_true, y_pm7)
rmsd_pm7 = np.sqrt(mean_squared_error(y_true, y_pm7))

print("======[ PM7 vs CAM-B3LYP ]======")
print(f" + Pearson r: {corr_pm7:.4f}")
print(f" + R^2: {r2_pm7:.4f}")
print(f" + MAE: {mae_pm7:.4f} eV")
print(f" + RMSD: {rmsd_pm7:.4f} eV\n")

corr_xtb, _ = pearsonr(y_xtb, y_true)
r2_xtb = r2_score(y_true, y_xtb)
mae_xtb = mean_absolute_error(y_true, y_xtb)
rmsd_xtb = np.sqrt(mean_squared_error(y_true, y_xtb))

print("======[ GFN2-xTB vs CAM-B3LYP ]======")
print(f" + Pearson r: {corr_xtb:.4f}")
print(f" + R^2: {r2_xtb:.4f}")
print(f" + MAE: {mae_xtb:.4f} eV")
print(f" + RMSD: {rmsd_xtb:.4f} eV")

======[ PM7 vs CAM-B3LYP ]======
 + Pearson r: 0.9214
 + R^2: -0.2609
 + MAE: 1.1055 eV
 + RMSD: 1.1938 eV

======[ GFN2-xTB vs CAM-B3LYP ]======
 + Pearson r: 0.8517
 + R^2: -14.7471
 + MAE: 4.1802 eV
 + RMSD: 4.2187 eV
